# Activity 09: Designing Helpful Functions

In [ ]:
# Just run this cell
from datascience import *
import numpy as np

Today you'll write small Python functions and hand them to a table with `.apply`, so the computer
does the same job on **1,177 rows** that you just did by hand on one.

A *helpful* function does one job, has a name that says what that job is, and gives the right
answer even on the awkward cases. You'll build five of them.

The data is every song that reached **#1** on the Billboard Hot 100 between August 1958 and
January 2025. Run the cell below to load it.

In [ ]:
songs = Table.read_table('data/billboard_top100.csv')
songs.show(3)

## Warm-up: what does `.apply` do?

A function takes something **in** and gives something back **out**. `.apply` runs your function on
every value in a column, one at a time, and collects the answers.

For each of the next four cells, **predict what will happen before you run it.** Say it out loud to
your partner first. The last one is broken on purpose.

In [ ]:
# Just run this cell
def shout(word):
    return word.upper() + "!"

shout("data")

In [ ]:
# Just run this cell. What comes back?
def quiet_shout(word):
    word.upper() + "!"

quiet_shout("data")

In [ ]:
# Just run this cell
demo = Table().with_columns('word', make_array('data', 'science', 'is', 'fun'))
demo.apply(shout, 'word')

In [ ]:
# Just run this cell -- it raises an error on purpose. Read the message carefully.
demo.apply(shout('word'))

## Trimming the table

Before you can think about a table, you have to be able to see it. Run the next two cells to find out
how big this one is.

In [ ]:
# Just run this cell
print(songs.num_rows, 'rows')
print(songs.num_columns, 'columns')

In [ ]:
# Just run this cell and skim the output
songs.labels

105 columns is too many to hold in your head, so the cell below keeps only the nine you need today.
It is done for you — `.select` is review from Lesson 05, and typing nine exact column names is not
what today is about.

In [ ]:
# Just run this cell
hits = songs.select('Song', 'Artist', 'Date', 'Weeks at Number One',
                    'Length (Sec)', 'Songwriters', 'BPM', 'Front Person Age', 'Explicit')
hits.show(5)

Those nine columns are:

|VARIABLE|DESCRIPTION|
|--------|-----------|
|Song|Data Type: String. The title of the song|
|Artist|Data Type: String. The performing artist|
|Date|Data Type: String. The week it reached #1, written like `4-Aug-58`|
|Weeks at Number One|Data Type: Integer. How many weeks it stayed at #1|
|Length (Sec)|Data Type: Integer. Song length in seconds|
|Songwriters|Data Type: String. Every credited writer, separated by semicolons|
|BPM|Data Type: Float. Tempo in beats per minute. Missing for 2 songs|
|Front Person Age|Data Type: Float. Age of the lead singer that year. Missing for 22 songs|
|Explicit|Data Type: Integer. 1 if the lyrics are explicit, 0 if not|

## Your first helpful function: `mmss`

`Length (Sec)` gives song length in seconds, which nobody thinks in. `213` seconds is really
**3:33**.

Write a function `mmss` that takes a number of seconds and returns a string like `'3:33'`. Two hints:
`//` gives you whole minutes and `%` gives you the leftover seconds, and `str(6).zfill(2)` turns
`6` into `'06'`.

**Test your function on a few values you can check by hand before you turn it loose on 1,177 rows.**
A wrong function applied to a whole column gives you 1,177 wrong answers and no error message.

In [ ]:
# You complete some code in this cell
def mmss(seconds):
    minutes   = ...
    remaining = ...
    return ...

In [ ]:
# Just run this cell -- check all three by hand
print(mmss(96))    # shortest #1 ever -- should be 1:36
print(mmss(613))   # longest  #1 ever -- should be 10:13
print(mmss(180))   # should be 3:00, not 3:0

Now run it on the whole column. **What type comes back?**

In [ ]:
# Just run this cell
hits.apply(mmss, 'Length (Sec)')

`.apply` handed back an **array** — the table itself hasn't changed. In the cell below:

1. Attach that array to `hits` as a new column called `Length`.
2. Show the 5 longest songs: sort by `Length (Sec)` descending, then select `Song`, `Artist` and
   `Length`.

**Discuss:** why sort by `Length (Sec)` rather than by your new `Length` column?

In [ ]:
# You complete some code in this cell
hits = ...
...

## What year was this a hit?

There is no `Year` column — only `Date`, which looks like `4-Aug-58`. The last two characters are the
year. Let's build the column we wish we had.

In [ ]:
# Just run this cell
hits.column('Date').take(np.arange(5))

Write a function `year` so that `year('4-Aug-58')` gives back the number `1958`.

In [ ]:
# You complete some code in this cell
def year(date):
    return ...

In [ ]:
# Just run this cell -- the oldest song in the data, then the newest
print(year('4-Aug-58'))
print(year('11-Jan-25'))

**Discuss:** what did `year('11-Jan-25')` give you? Is that right?

The data covers 1958 through 2025, but a two-digit year cannot say which century it belongs to. Under
the rule you just wrote, **327 of these 1,177 songs land in the wrong century.**

Fix `year` below so both centuries come out right — and be ready to say *what cutoff you chose and
why that number*.

In [ ]:
# You complete some code in this cell
def year(date):
    yy = int(date[-2:])
    ...

Check the fix, then attach a `Year` column to `hits` and show the 3 **oldest** songs
(`Song`, `Artist`, `Year`).

In [ ]:
# You complete some code in this cell
print(year('4-Aug-58'), year('11-Jan-25'))

hits = ...
...

In [ ]:
# Just run this cell -- a sanity check. Oldest should be 1958, newest 2025.
print(min(hits.column('Year')), max(hits.column('Year')))

## How many people does it take to write a #1 hit?

The `Songwriters` column lists every credited writer, separated by semicolons:

`'John Lennon;Paul McCartney'`

That's a *string*, not a count. To count the writers you have to split it apart.

In [ ]:
# Just run this cell
hits.column('Songwriters').take(np.arange(4))

In [ ]:
# Just run this cell -- predict the output first
'John Lennon;Paul McCartney'.split(';')

Write `count_writers` so that `count_writers('John Lennon;Paul McCartney')` gives `2`.

In [ ]:
# You complete some code in this cell
def count_writers(names):
    return ...

In [ ]:
# Just run this cell
print(count_writers('Lionel Richie'))                              # 1
print(count_writers('John Lennon;Paul McCartney'))                 # 2
print(count_writers('Brian Holland;Lamont Dozier;Eddie Holland'))  # 3

Attach a `Writers` column to `hits`, then answer: which #1 hit had the biggest writing committee?
Sort by `Writers` descending and select `Song`, `Artist`, `Year`, `Writers`.

In [ ]:
# You complete some code in this cell
hits = ...
...

**The real question.** Has the number of people it takes to write a #1 hit changed over time?

You now have a `Year` column and a `Writers` column, neither of which existed ten minutes ago. The
cell below works out the average for 1958–1969. Use it as your pattern.

In [ ]:
# Just run this cell
early = hits.where('Year', are.between_or_equal_to(1958, 1969))
np.mean(early.column('Writers'))

Now do the same for 2010–2025. **Predict first:** bigger, smaller, or about the same?

In [ ]:
# You complete some code in this cell
late = ...
...

**Discuss:** why might that number have changed so much? Careful — the data can tell you *that* it
changed, but not *why*. What would you need to know to actually answer the why?

*(You just compared two eras by hand. Tonight's lesson — `group` — does all seven decades at once, in
one line.)*

## Sorting values into categories

Functions don't have to return numbers. A function that returns a **label** turns a numeric column
into a categorical one — exactly what you need before you can group or compare.

Write `tempo_label` so that anything under 90 BPM is `'slow'`, 90 up to and including 120 is
`'medium'`, and above 120 is `'fast'`.

In [ ]:
# You complete some code in this cell
def tempo_label(bpm):
    if ...:
        return 'slow'
    elif ...:
        return 'medium'
    else:
        return 'fast'

In [ ]:
# Just run this cell -- should print: slow medium fast
print(tempo_label(72), tempo_label(120), tempo_label(155))

Attach a `Tempo` column to `hits`, then finish the counts. The first one shows you the pattern.

In [ ]:
# You complete some code in this cell
hits = ...

print('slow  ', hits.where('Tempo', 'slow').num_rows)
print('medium', ...)
print('fast  ', ...)

## Functions that need more than one column

`.apply` can hand your function values from **several** columns at once. You list the column names in
the same order as the function's arguments:

```python
table.apply(my_function, 'First Column', 'Second Column')
```

Write `writers_per_minute`, which takes a songwriter string and a length in seconds, and returns how
many writers there are per minute of song.

In [ ]:
# You complete some code in this cell
def writers_per_minute(names, seconds):
    return ...

The `.apply` call is done for you below — note the two column names. Your job is the second line: show
the 5 songs with the most writers per minute (`Song`, `Artist`, `Writers`, `Length`).

In [ ]:
# You complete some code in this cell
hits = hits.with_column('Writers/Min',
                        hits.apply(writers_per_minute, 'Songwriters', 'Length (Sec)'))
...

## When your function meets missing data

`Front Person Age` is missing for 22 songs. Missing numbers arrive as `nan` ("not a number").
Run the next two cells and watch what `nan` does to an innocent-looking function.

In [ ]:
# Just run this cell -- look for the nan values
hits.select('Song', 'Front Person Age').show(8)

In [ ]:
# Just run this cell -- predict all three answers first
def is_young(age):
    if age < 25:
        return 'under 25'
    else:
        return '25 or older'

print(is_young(19))
print(is_young(40))
print(is_young(np.nan))

**Discuss:** `is_young(np.nan)` didn't crash. It quietly answered `'25 or older'`.

Every comparison against `nan` is `False`, so the `if` fell through to the `else`. Applied to this
table, that silently sorts 22 songs with *unknown* singer ages into the "25 or older" pile. No error,
no warning, just a wrong answer buried in a column.

Rewrite `is_young` so missing ages come back labelled `'unknown'` instead. `np.isnan(age)` tells you
whether a value is missing.

In [ ]:
# You complete some code in this cell
def is_young(age):
    ...

Check the fix, then attach an `Age Group` column and count how many songs came back `'unknown'`.
You should get 22.

In [ ]:
# You complete some code in this cell
print(is_young(19), '|', is_young(40), '|', is_young(np.nan))

hits = ...
print('unknown ages:', ...)

## Wrap-up

| Tool | What it does |
| --- | --- |
| `def name(x):` … `return` | define a function: something in, something out |
| `table.apply(f, 'Col')` | run `f` on every value of `Col`, return an array of results |
| `table.apply(f, 'A', 'B')` | run `f` on each row's `A` and `B` together |
| `table.with_column('New', values)` | attach the results as a new column |
| `text.split(';')` | break a string into a list on a separator |
| `str(n).zfill(2)` | pad a number to two digits: `6` → `'06'` |
| `if` / `elif` / `else` | return a different answer for each case |
| `np.isnan(x)` | check for missing numeric data before you compare it |

**The two habits to keep:**

1. Test a function on values you can check by hand *before* applying it to the whole table.
2. `.apply` gives you back an **array**. The table doesn't change until you `with_column` it.